# Práctica — Clase 8: resume la transcripción con un modelo gratuito

En el proyecto real "AI Meeting Companion" de IBM, después de transcribir el audio con
Whisper, el texto se le pasa a un LLM con un prompt del estilo "*List the key points with
details from the context*" para extraer los puntos clave de la reunión. El material original
usa un modelo de pago de IBM watsonx para ese paso. En esta práctica vas a resolver el mismo
caso de uso —transcribir y resumir una reunión— usando un modelo de resumen 100% gratuito de
Hugging Face.

## Ejercicio — Completa `summarize_meeting()`

Ya tienes lista la transcripción del audio de ejemplo (igual que en el notebook de clase). Tu
tarea es completar la función `summarize_meeting()` para que, dado el texto transcrito,
devuelva un resumen con los puntos clave, usando un pipeline de resumen de Hugging Face (por
ejemplo `sshleifer/distilbart-cnn-12-6`, una versión más liviana de `facebook/bart-large-cnn`).

In [ ]:
!pip install -q transformers torch requests

In [ ]:
import requests
from transformers import pipeline

# 1) Descargamos el mismo audio de ejemplo de la clase
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-GPXX04C6EN/Testing%20speech%20to%20text.mp3"
response = requests.get(url)
audio_file_path = "downloaded_audio.mp3"
with open(audio_file_path, "wb") as file:
    file.write(response.content)

# 2) Transcribimos el audio con Whisper, igual que en el notebook de clase
asr_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny.en",
    chunk_length_s=30,
)
transcript_txt = asr_pipe(audio_file_path, batch_size=8)["text"]
print("Transcripción:")
print(transcript_txt)

In [ ]:
# TODO 1: crea un pipeline de resumen ("summarization") con un modelo gratuito
#         de Hugging Face, por ejemplo "sshleifer/distilbart-cnn-12-6"
summarizer = None  # <-- reemplaza esto

def summarize_meeting(text):
    # TODO 2: usa el summarizer para generar un resumen del texto.
    #         Pista: summarizer(text, max_length=60, min_length=15, do_sample=False)
    #         La salida es una lista de diccionarios con la clave "summary_text".
    #         Devuelve el string del resumen.
    pass  # <-- reemplaza esto

# Descomenta para probar una vez completados los TODO:
# print(summarize_meeting(transcript_txt))

<details>
<summary>💡 Ver solución</summary>

```python
from transformers import pipeline

summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

def summarize_meeting(text):
    resultado = summarizer(text, max_length=60, min_length=15, do_sample=False)
    return resultado[0]["summary_text"]

print(summarize_meeting(transcript_txt))
```

Nota: si el texto transcrito es muy corto, algunos modelos de resumen pueden fallar o
devolver casi el mismo texto — es normal, los modelos de resumen están pensados para textos
más largos, como una reunión completa. También puedes probar con `facebook/bart-large-cnn`
en vez de `sshleifer/distilbart-cnn-12-6`: es más preciso, pero más lento y pesado.
</details>

## Referencia: así se hace con el LLM de pago del material original (watsonx)

El proyecto real `AIMeetingCompanion_Project/speech_analyzer.py` encadena la transcripción
con un LLM de IBM watsonx usando LangChain:

```python
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from ibm_watson_machine_learning.foundation_models.extensions.langchain import WatsonxLLM

temp = '''
<s><<SYS>>
List the key points with details from the context:
[INST] The context : {context} [/INST]
<</SYS>>
'''
pt = PromptTemplate(input_variables=["context"], template=temp)
prompt_to_LLAMA2 = LLMChain(llm=llm, prompt=pt)  # llm = WatsonxLLM(...)

result = prompt_to_LLAMA2.run(transcript_txt)
```

Esta vía requiere credenciales de pago de IBM watsonx; por eso, en esta práctica resolviste
el mismo caso de uso con un modelo de resumen gratuito de Hugging Face.

Vuelve al [notebook de clase](../notebooks_clase/clase08_whisper_voz.ipynb) o repasa el
[Capítulo 12 del manual](../manual/12_whisper_voz.md).